In [ ]:
try:
    import claveles  # noqa: F401
except ImportError:
    %pip install claveles

**Note for Google Colab Users:**
During installation, Colab may display a dependency conflict error regarding `protobuf` (`ERROR: pip's dependency resolver...`). This occurs due to pre-installed Google packages in the Colab environment. 

As long as you see `Successfully installed ...` at the end of the log, the required packages are properly installed. **Please ignore the error and continue running the rest of the notebook.**

$$
\newcommand{\bra}[1]{{\left\langle{#1}\right|}}
\newcommand{\ket}[1]{{\left|{#1}\right\rangle}}
\newcommand{\Bra}[1]{{\Big\langle{#1}\Big|}}
\newcommand{\Ket}[1]{{\Big|{#1}\Big\rangle}}
\newcommand{\braket}[1]{{\left\langle{#1}\right\rangle}}
\newcommand{\dag}{\dagger}
$$

# Fock-Number Measurement on an EPR State: Heralding Non-Gaussian States

## Introduction

The first notebook generated EPR states and evaluated their quadrature correlations using homodyne detection—a Gaussian measurement that preserves the Gaussian nature of the conditional state. In this tutorial, we replace homodyne detection with **Fock-number measurement**, which is a non-Gaussian projective measurement. By simulating how such non-Gaussian measurement produces a conditioned state in the residual mode, we demonstrate that the EPR state serves as a source of *heralded Fock states*—non-Gaussian states that cannot be produced by any Gaussian operation.

> **Prerequisite:** This notebook builds directly upon [Two-Mode EPR State: Generation and Quantum Correlations](./epr1.ipynb), which covers the fundamentals of EPR state generation and correlations. We highly recommend reading that notebook first.
>
> This is one of two independent follow-up notebooks; the other, [Generating EPR States with a CZ Gate](./epr2.ipynb), provides a quantitative comparison between the beamsplitter and CZ-gate generation methods. Both follow-up notebooks depend exclusively on the foundational first notebook and can be read in any order.

---

## Background

### Recap: The EPR State

We construct a two-mode squeezed vacuum in the Fock representation by mixing two single-mode squeezed vacua on a 50:50 beamsplitter, identical to the procedure in the first notebook. Here we employ the Fock backend for numerical simulation of a CV circuit equipped with Fock-number counting measurement, which is one of the most fundamental and resourceful non-Gaussian processes in CV quantm computing.

### Generating Non-Gaussian States by Fock-Number Measurement

Homodyne detection is a Gaussian operation: when applied to a Gaussian state, it yields a Gaussian conditional state. In contrast, Fock-number measurement (associated with the number operator $\hat{n}$) is fundamentally non-Gaussian. This is because the Fock states $\ket{n}$ for $n\geq 1$ are inherently non-Gaussian, and thus a measurement that projects a qumode onto such non-Gaussian states can herald another non-Gaussian state in the partner mode. The non-Gaussianity of the unmeasured partner mode can be characterized by *Wigner negativity* (regions where the Wigner quasi-probability distribution takes negative values, which is impossible to achieve via Gaussian operations).

In the Fock basis, the finite-energy two-mode squeezed vacuum state $\ket{\mathrm{EPR}_r}$ is expressed as below (see [Appendix](#appendix-epr-state-in-fock-basis) for detailed deriviation):

$$
\ket{\mathrm{EPR}_r} = \sqrt{1 - \lambda^2}\sum_{n=0}^{\infty}\lambda^n\,\ket{n}_0\otimes\ket{n}_1,
\qquad \lambda = \tanh r.
$$

This state is perfectly correlated in the Fock-number basis; the superposed Fock states $\ket{n}_0\otimes\ket{n}_1$ always contain an identical number of bosons in both modes. Projecting mode 0 onto a Fock-$n$ state via the projection operator $\hat{E} = \ket{n}\bra{n}\otimes\hat{I}$ and subsequently tracing it out leaves mode 1 in the state:

$$
\hat{\rho}'_1 = \frac{\operatorname{tr}_0(\hat{E}\,\hat{\rho}\,\hat{E})}{\operatorname{tr}(\hat{E}\,\hat{\rho})} = \ket{n}\!\bra{n}_1,
$$

which corresponds exactly to the same Fock state $\ket{n}$, a distinct non-Gaussian state when $n\geq 1$. Consequently, detecting $n$ bosons in mode 0 *heralds* the generation of exactly $n$ bosons in mode 1. (In the ideal limit where $r \to \infty$ and $\lambda \to 1$, this recovers the equal-weight superposition of $x$-quadrature eigenstates that defines the ideal EPR state.)

---

## 1. Setup

First, let us import the required libraries, define the common simulation parameters, and construct the beamsplitter-based EPR state in the Fock representation.

In [ ]:
from math import pi

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from claveles.circuit import CircuitRepr
from claveles.circuit.ops.std import BeamSplitter, MeasureFock
from claveles.circuit.state import QuantumState
from claveles.client import SimulatorClient
from claveles.execute import execute
from numpy import linspace

In [ ]:
r = 1.0  # squeezing parameter
cutoff_dim = 16  # Fock-space dimension D
n_shots = 2000

xvec = linspace(-5, 5, 121)
pvec = linspace(-5, 5, 121)

sns.set_theme(style="whitegrid")

As in the first notebook, we place an $x$-squeezed vacuum on mode 0 and a $p$-squeezed vacuum on mode 1 and combine them on the beamsplitter $\hat{B}(\pi/4, \pi)$ using `BeamSplitter(theta=pi / 4, phi=pi)`. The squeezed vacua are built directly in the Fock basis with `QuantumState.squeezed(..., repr="fock", cutoff_dim=cutoff_dim)`, truncated to Fock numbers below $D$. The output is the two-mode squeezed vacuum $\ket{\mathrm{TMSV}}$ of the Background with $\lambda = \tanh r$, which carries the perfect Fock-number correlation this notebook relies on.

In [ ]:
def build_epr_bs(
    r: float, cutoff_dim: int, name: str = "EPR via BeamSplitter"
) -> CircuitRepr:
    c = CircuitRepr(name)
    # x-squeezed vacuum on mode 0, p-squeezed vacuum on mode 1
    c.Q(0) | QuantumState.squeezed(r=r, phi=0, repr="fock", cutoff_dim=cutoff_dim)
    c.Q(1) | QuantumState.squeezed(
        r=r, phi=pi / 2, repr="fock", cutoff_dim=cutoff_dim
    )
    c.Q(0, 1) | BeamSplitter(theta=pi / 4, phi=pi)
    return c

In [ ]:
client_shots = SimulatorClient(n_shots=n_shots, backend="cpu", state_save_policy="none")
client_cond = SimulatorClient(n_shots=1, backend="cpu", state_save_policy="first_only")

---

## 2. Fock-Number Correlation

We evaluate the Fock-number statistics across both modes to confirm that the measurement outcomes always coincide. A small fraction of shots disagrees in the simulation—this is because each squeezed input is truncated to $D$ Fock levels before the beamsplitter, and the simulated state therefore deviates slightly from the ideal one. Note that the deviation shrinks as one expands $D$.

In [ ]:
def build_epr_bs_fock(r: float, cutoff_dim: int) -> CircuitRepr:
    c = build_epr_bs(r=r, cutoff_dim=cutoff_dim, name="EPR BS Fock-number")
    c.Q(0) | MeasureFock()  # Fock-number measurement on mode 0
    c.Q(1) | MeasureFock()  # Fock-number measurement on mode 1
    return c

In [ ]:
circuit_fock = build_epr_bs_fock(r=r, cutoff_dim=cutoff_dim)
result_fock = execute(circuit_fock, client_shots)

n0 = np.array(
    [result_fock.client_result[i].get_value(0) for i in range(n_shots)], dtype=int
)
n1 = np.array(
    [result_fock.client_result[i].get_value(1) for i in range(n_shots)], dtype=int
)

print(f"Fraction of shots with n0 == n1: {np.mean(n0 == n1):.3f}  (ideal: 1.0)")

To visualize the correlations directly, we plot the joint outcomes $(n_0, n_1)$ as a heatmap. The axes are restricted to Fock numbers that occur with non-negligible frequency, as high-$n$ outcomes are rare due to the small mean Fock number.

In [ ]:
n_max = int(np.percentile(np.concatenate([n0, n1]), 99.5))
joint = np.zeros((n_max + 1, n_max + 1))
for a, b in zip(n0, n1):
    if a <= n_max and b <= n_max:
        joint[a, b] += 1
joint /= n_shots

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    joint,
    annot=True,
    fmt=".2f",
    cmap="rocket_r",
    square=True,
    cbar_kws={"label": "probability"},
    ax=ax,
)
ax.set_xlabel(r"$n_1$")
ax.set_ylabel(r"$n_0$")
ax.invert_yaxis()
ax.set_title(f"Joint Fock-number distribution (BS method, $r = {r}$)")
fig.tight_layout()
plt.show()

The entire probability distribution concentrates along the diagonal $n_0 = n_1$: detecting $n$ bosons in mode 0 guarantees the presence of exactly the same $n$ bosons in mode 1. This serves as the Fock-number analogue of the $\hat{x}_0 \approx \hat{x}_1$ correlation observed via homodyne detection in the first notebook; both phenomena stem from the same underlying EPR entanglement.

---

## 3. Heralded Fock State

To observe the heralded non-Gaussian state directly, we measure only mode 0 and visualize the conditional Wigner function of mode 1. By post-selecting the measurement on a fixed Fock number $n$ (via `MeasureFock(post_select=n)`), we implement the projection operation $\hat{E} = \ket{n}\bra{n}\otimes\hat{I}$ described in the background, leaving mode 1 in the heralded Fock state $\ket{n}$. Setting `remove=True` discards the measured mode 0, isolating mode 1 as the sole remaining mode.

In [ ]:
herald_n = 2  # post-select mode 0 on this Fock number


def build_epr_bs_herald(r: float, cutoff_dim: int, n: int) -> CircuitRepr:
    c = build_epr_bs(r=r, cutoff_dim=cutoff_dim, name="EPR BS herald")
    c.Q(0) | MeasureFock(post_select=n)  # herald mode 1 on n Fock state
    return c

In [ ]:
circuit_herald = build_epr_bs_herald(r=r, cutoff_dim=cutoff_dim, n=herald_n)
result_herald = execute(circuit_herald, client_cond)

# Mode 0 was consumed by the measurement, so the state holds mode 1 alone.
state_mode1 = result_herald.client_result.shot_results[0].state

print(f"Post-selected Fock number on mode 0 (heralding mode 1): n = {herald_n}")

fig = state_mode1.draw_wigner(
    xvec, pvec, title=f"Conditional state of mode 1 (heralded n = {herald_n})"
)
fig.tight_layout()
plt.show()

The conditional Wigner function exhibits the distinct concentric-ring structure characteristic of the Fock state $\ket{n}$ for the exact Fock number $n$ post-selected on mode 0. Crucially, it displays regions of negative quasi-probability, providing an unambiguous signature of a non-Gaussian state. This serves as the conditional-state counterpart to the perfect Fock-number correlation discussed in Section 2. While the homodyne measurement demonstrated in the first notebook leaves the partner mode in a Gaussian displaced squeezed state, heralding via Fock-number measurement generates a quantum state that lies entirely outside the Gaussian family. Note that, however, such Fock-state generation is always conditional depending on the probabilistic measurement outcome $n$ on mode 0, thus the process alone cannot directly be exploited for deterministic state engineering.

---

## Summary

This tutorial demonstrated the following:

1. **Fock-number correlation**: Performing Fock-number measurements on both modes of an EPR state consistently yields identical outcomes ($n_0 = n_1$), serving as the Fock-number analogue to the homodyne quadrature correlation $\hat{x}_0 \approx \hat{x}_1$.
2. **Heralded Fock states**: Measuring the Fock number of one mode successfully heralds a specific Fock state $\ket{n}$ on the companion mode, which is a non-Gaussian state characterized by pronounced Wigner negativity.
3. **Gaussian vs. non-Gaussian conditioning**: Conditioning on a homodyne measurement (a Gaussian operation) results in a displaced squeezed state, whereas heralding via Fock-number measurement (a non-Gaussian operation) produces a state with clear Wigner negativity, which cannot be achieved through any combination of Gaussian operations.

---

## Appendix: EPR state in Fock basis

In this section, we derive the Bogoliubov-transformed annihilation operators (nullifiers) for the EPR state $\ket{\text{EPR}_r}$, generated by mixing an $x$-squeezed state and a $p$-squeezed state with the same squeezing parameter on a 50:50 beamsplitter, and use them to mathematically deduce the exact Fock expansion of the state.

### 1. Deriving the Nullifiers from Beamsplitter Interference

Consider two independent single-mode squeezed vacua prepared at the input ports 0 and 1. To achieve the correct EPR correlations (i.e. $\hat{x}_0-\hat{x}_1\approx0,\quad\hat{p}_0+\hat{p}_1\approx0$), we configure the inputs as follows:
- **Mode 0:** An $x$-squeezed state $\ket{r}$ (squeezing parameter $r > 0$). Its nullifier is $\hat{c}_0 = \hat{a}_0 \cosh r + \hat{a}_0^\dagger \sinh r$. By substituting $\hat{a} = \frac{1}{\sqrt{2\hbar}}(\hat{x} + i\hat{p})$, this expands to $\hat{c}_0 = \frac{1}{\sqrt{2\hbar}} (e^r \hat{x}_0 + i e^{-r} \hat{p}_0)$.
- **Mode 1:** A $p$-squeezed state $\ket{-r}$ (squeezing parameter $r > 0$, rotated by $\pi/2$). Its nullifier is $\hat{c}_1 = \hat{a}_1 \cosh r - \hat{a}_1^\dagger \sinh r$. This expands to $\hat{c}_1 = \frac{1}{\sqrt{2\hbar}} (e^{-r} \hat{x}_1 + i e^r \hat{p}_1)$.

Here, an operator $\hat{c}$ (which is a linear combination of quadrature operators) is called a *nullifier* of a pure Gaussian state $\ket{\psi}$ when $\hat{c}\ket{\psi}=0$ holds. These initial states (jointly represented as a tensor product of the squeezed states as $\ket{\text{in}} = \ket{r}_0 \otimes \ket{-r}_1$) are annihilated by their respective nullifiers, meaning $\hat{c}_0 \ket{\text{in}} = 0$ and $\hat{c}_1 \ket{\text{in}} = 0$. Note that in the limit of $r\to\infty$, the nullifier $\hat{c}_0$ is approximately proportional to $\hat{x}$, confirming that $\hat{x}\ket{r}\approx 0$ holds—i.e. the state in mode 0 is squeezed along the position quadrature and concentrated around $x=0$ (and the same for mode 1 but in the momentum quadrature).

We mix these two modes using a 50:50 beamsplitter defined by the unitary operator $\hat{B}(\pi/4, \pi) = \exp\left[\frac{\pi}{4}(-\hat{a}_0\hat{a}_1^\dagger + \hat{a}_0^\dagger\hat{a}_1)\right]$. The output state is $\ket{\text{EPR}_r} = \hat{B}(\pi/4, \pi) \ket{\text{in}}$.
The nullifiers $\hat{c}'_0$ and $\hat{c}'_1$ for the output state $\ket{\text{EPR}_r}$ can be found using the following equations:

$$
\hat{c}_0 \ket{\text{in}} = 0 \quad\iff \hat{B}\hat{c}_0 \hat{B}^\dagger\hat{B}\ket{\text{in}} = 0\quad\iff \hat{B}\hat{c}_0 \hat{B}^\dagger\ket{\text{EPR}_r} = 0
$$

$$
\hat{c}_1 \ket{\text{in}} = 0 \quad\iff \hat{B}\hat{c}_1 \hat{B}^\dagger\hat{B}\ket{\text{in}} = 0\quad\iff \hat{B}\hat{c}_1 \hat{B}^\dagger\ket{\text{EPR}_r} = 0
$$

in which we employed the unitarity of the beamsplitter gate $\hat{B}^\dagger \hat{B} = \hat{I}$. Thus, the nullifiers of a quantum state are transformed as the inverse of normal Heisenberg transformation. By applying the inverse conjugation $\hat{c}' = \hat{B}\,\hat{c}\,\hat{B}^\dagger$, the mode operators transform as:

$$
\hat{B}\,\hat{a}_0\,\hat{B}^\dagger = \frac{\hat{a}_0 - \hat{a}_1}{\sqrt{2}}, \qquad
\hat{B}\,\hat{a}_1\,\hat{B}^\dagger = \frac{\hat{a}_0 + \hat{a}_1}{\sqrt{2}}.
$$

Applying this to the initial nullifiers gives the transformed operators $\hat{c}_0'$ and $\hat{c}_1'$ as:

$$
\hat{c}_0' = \frac{\hat{a}_0 - \hat{a}_1}{\sqrt{2}}\cosh r + \frac{\hat{a}_0^\dagger - \hat{a}_1^\dagger}{\sqrt{2}}\sinh r = \frac{1}{2\sqrt{\hbar}} \left[ e^r (\hat{x}_0 - \hat{x}_1) + i e^{-r} (\hat{p}_0 - \hat{p}_1) \right]
$$

$$
\hat{c}_1' = \frac{\hat{a}_0 + \hat{a}_1}{\sqrt{2}}\cosh r - \frac{\hat{a}_0^\dagger + \hat{a}_1^\dagger}{\sqrt{2}}\sinh r = \frac{1}{2\sqrt{\hbar}} \left[ e^{-r} (\hat{x}_0 + \hat{x}_1) + i e^r (\hat{p}_0 + \hat{p}_1) \right]
$$

Again, the nullifiers approximately behave as $\hat{c}_0' \propto x_0 - x_1$ and $\hat{c}_1' \propto p_0 + p_1$ in the limit of $r \to \infty$, guaranteeing the position correlation and the momentum anti-correlation of the EPR state (i.e. $\Delta(\hat{x}_0 - \hat{x}_1) \approx 0$ and $\Delta(\hat{p}_0 + \hat{p}_1) \approx 0$.)

To simplify the math, we construct a new pair of nullifiers by taking the symmetric and antisymmetric linear combinations of $\hat{c}_0'$ and $\hat{c}_1'$:

**The sum of the nullifiers:**
$$
\hat{b}_0 = \frac{1}{\sqrt{2}}\left(\hat{c}_0' + \hat{c}_1'\right) = \hat{a}_0 \cosh r - \hat{a}_1^\dagger \sinh r
$$

**The difference of the nullifiers:**
$$
\hat{b}_1 = \frac{1}{\sqrt{2}}\left(\hat{c}_1' - \hat{c}_0'\right) = \hat{a}_1 \cosh r - \hat{a}_0^\dagger \sinh r
$$

These derived operators $\hat{b}_0$ and $\hat{b}_1$ are the Bogoliubov-transformed annihilation operators, satisfying $\hat{b}_0\ket{\text{EPR}_r} = 0$ and $\hat{b}_1\ket{\text{EPR}_r} = 0$.

### 2. Deriving the Fock Expansion

Now that we have established the nullifiers $\hat{b}_0$ and $\hat{b}_1$, we can determine the exact form of $\ket{\text{EPR}_r}$ in the Fock number basis. Let us express the state as a general superposition of two-mode Fock states without assuming any prior correlation:

$$
\ket{\text{EPR}_r} = \sum_{n=0}^{\infty} \sum_{m=0}^{\infty} c_{n,m} \ket{n}_0 \ket{m}_1
$$

Applying the first nullifier condition, $\hat{b}_0 \ket{\text{EPR}_r} = 0$:

$$
(\hat{a}_0 \cosh r - \hat{a}_1^\dagger \sinh r) \sum_{n=0}^{\infty} \sum_{m=0}^{\infty} c_{n,m} \ket{n}_0 \ket{m}_1 = 0
$$

Applying the creation and annihilation operators to the Fock basis yields:

$$
\sum_{n=1}^{\infty} \sum_{m=0}^{\infty} c_{n,m} \sqrt{n} \cosh r \ket{n-1}_0 \ket{m}_1 - \sum_{n=0}^{\infty} \sum_{m=0}^{\infty} c_{n,m} \sqrt{m+1} \sinh r \ket{n}_0 \ket{m+1}_1 = 0
$$

To find the relations between coefficients, we compare the terms that result in a specific basis state $\ket{k}_0\ket{l}_1$.
For the first sum, a term results in $\ket{k}_0\ket{l}_1$ when $n-1 = k$ and $m = l$, meaning $n = k+1$ and $m = l$. This term exists for all $l \ge 0$.
For the second sum, a term results in $\ket{k}_0\ket{l}_1$ when $n = k$ and $m+1 = l$, meaning $n = k$ and $m = l-1$. Since $m$ cannot be negative, this term only exists when $l \ge 1$.

For $l \ge 1$, equating the coefficients from both sums to zero yields the recurrence relation:

$$
c_{k+1, l} \sqrt{k+1} \cosh r - c_{k, l-1} \sqrt{l} \sinh r = 0 \implies c_{k+1, l} = c_{k, l-1} \sqrt{\frac{l}{k+1}} \tanh r \quad \text{--- (Eq. A)}
$$

Applying the second nullifier condition, $\hat{b}_1 \ket{\text{EPR}_r} = 0$, and following the exact same logic for the other mode, we obtain the complementary recurrence relation valid for $k \ge 1$:

$$
c_{k, l+1} = c_{k-1, l} \sqrt{\frac{k}{l+1}} \tanh r \quad \text{--- (Eq. B)}
$$

### 3. Perfect Correlation in the Fock basis

The conditions derived above enforce the perfect Fock number correlation of the EPR state. Let us look at the boundary conditions where $l=0$.
For the basis state $\ket{k}_0\ket{0}_1$ (i.e., $l=0$), the second sum in our $\hat{b}_0$ derivation has no contribution because the term $\ket{m+1}_1$ with $m \geq 0$ cannot yield the vacuum state $\ket{0}$ in mode 1. Therefore, only the first sum contributes, directly demanding that:
$$
c_{k+1, 0} \sqrt{k+1} \cosh r = 0 \implies c_{k+1, 0} = 0 \quad (\text{for all } k \ge 0)
$$
Similarly, applying the $k=0$ boundary condition to the $\hat{b}_1$ derivation yields $c_{0, l+1} = 0$.

By induction, this domino effect dictates that all off-diagonal coefficients where $k \neq l$ are strictly zero. To see why, consider applying the recurrence relations repeatedly. If we start with an asymmetric term where $k > l$, reducing both indices step-by-step will eventually hit $l=0$ while $k-l$ is still positive. This traces the computation back to the boundary condition $c_{k-l, 0} = 0$, forcing the entire coefficient to be zero. Conversely, if $k < l$, the indices will hit $k=0$ first, tracing back to $c_{0, l-k} = 0$ and again resulting in zero. In contrast, the diagonal terms where $k=l$ survive, because they reduce down to $c_{0,0}$ simultaneously—representing the non-zero probability amplitude of the two-mode vacuum state. Thus, the mathematics itself forces the Fock numbers to be perfectly paired ($n=m$).

To find the recurrence relation for the surviving diagonal terms, we set $l = k+1$ in (Eq. A) so that both indices match:

$$
c_{k+1, k+1} = c_{k, k} \sqrt{\frac{k+1}{k+1}} \tanh r = c_{k, k} \tanh r
$$

Letting $c_{n,n} \equiv c_n$, this simple recurrence relation $c_n = c_{n-1} \tanh r$ dictates that:

$$
c_n = c_0 (\tanh r)^n
$$

Finally, we determine $c_0$ using the normalization condition $\sum_{n=0}^{\infty} |c_n|^2 = 1$:

$$
\sum_{n=0}^{\infty} |c_0|^2 (\tanh r)^{2n} = |c_0|^2 \frac{1}{1 - \tanh^2 r} = |c_0|^2 \cosh^2 r = 1 \implies c_0 = \frac{1}{\cosh r}
$$

Substituting this back, we arrive at the exact Schmidt decomposition of the continuous-variable EPR state:

$$
\ket{\text{EPR}_r} = \frac{1}{\cosh r} \sum_{n=0}^{\infty} (\tanh r)^n \ket{n}_0 \ket{n}_1
$$

---

## References

- \[1\] A. Einstein, B. Podolsky, and N. Rosen, Can Quantum-Mechanical Description of Physical Reality Be Considered Complete?, [*Phys. Rev.* **47**, 777 (1935)](https://doi.org/10.1103/PhysRev.47.777).
- \[2\] J. S. Bell, On the einstein podolsky rosen paradox, [*Phys. Phys. Fiz* **1**, 195 (1964)](https://doi.org/10.1103/PhysicsPhysiqueFizika.1.195).
- \[3\] A. Heidmann, R. J. Horowicz, S. Reynaud, E. Giacobino, C. Fabre, and G. Camy, Observation of quantum noise reduction on twin laser beams, [*Phys. Rev. Lett.* **59**, 2555 (1987)](https://doi.org/10.1103/PhysRevLett.59.2555).